# 05 - Persona Scoring Model

Core persona methodology, vectorization, spatial join, full model scoring, and internal-only scoring.

> Split from `PersonaMatch.ipynb` without re-running cells; saved outputs are preserved as stored in the original notebook.


**The Persona-Based Rating Model - Core Methodology:**

In [0]:
from pyspark.sql.functions import concat_ws, col 

# 1. Final text cleanup - concatenating lists into strings for the UDF
# Our UDF expects strings, not arrays
scoring_base_df = internal_df.withColumn("amenities_str", concat_ws(", ", col("amenities_list"))) \
                             .withColumn("highlights_str", concat_ws(", ", col("highlights_list"))) \
                             .withColumn("reviews_str", concat_ws(" || ", col("reviews")))

# 2. Removing duplicate assets (if there is an asset that appears twice by mistake)
unique_listings = scoring_base_df.dropDuplicates(['listing_name', 'lat', 'long'])

**Setup + Vectorization + Optimized Spatial Join:**

In [0]:
# CELL 1 — Data Preparation + Vectorization + Spatial Join:
# 1) Builds config and semantic lexicons (Word2Vec centroids)
# 2) Broadcasts config/model to executors
# 3) Vectorizes places + listings (text -> vectors)
# 4) Performs optimized spatial join (grid bucketing + bbox + haversine)
# 5) Aggregates nearby context into a single centroid vector per listing
# Output: final_prep_df (ready for scoring)

from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, lit, acos, cos, sin, radians, udf, collect_list, count,
    when, greatest, floor, explode, array, struct, abs as ps_abs
)
from pyspark.sql.types import DoubleType, StringType, MapType, ArrayType
import numpy as np
import math

# Global constants
RADIUS_KM = 1.5
EARTH_KM = 6371.0

# 1) CONFIGURATION & BROADCAST
full_config_data = {
    "text_lexicons": user_text_lexicons,
    "amenities_map": {
        "singles": ["wifi", "self check-in", "lockbox", "gym", "kitchen"],
        "couples": ["hot tub", "jacuzzi", "king size bed", "patio", "balcony", "sound system", "dimmable lighting"],
        "families": ["crib", "high chair", "baby bath", "pack ’n play/travel crib", "changing table",
                     "children’s books and toys", "bathtub", "washing machine"],
        "remote_workers": ["wifi", "dedicated workspace", "ethernet", "office chair", "coffee maker", "keypad"],
    },
    "highlights_map": {
        "singles": {"self check-in": 5.0, "vibrant neighbourhood": 4.0, "superhost": 2.0, "great location": 4.0, "in center": 4.0},
        "couples": {"superhost": 3.0, "great location": 4.0, "self check-in": 2.0, "couples": 5.0, "romantic": 5.0,"dining outside": 3.0},
        "families": {"families": 5.0, "superhost": 4.0, "free parking": 3.0, "kitchen": 3.0, "self check-in": 2.0, "kids": 5.0, "quiet": 4.0},
        "remote_workers": {"fast wifi": 6.0, "workspace": 6.0, "self check-in": 5.0, "superhost": 2.0, "quiet": 4.0},
    }
}

def get_phrase_vector(phrase_list, model):
    all_phrase_vectors = []
    for phrase in phrase_list:
        words = str(phrase).lower().replace(",", "").split()
        word_vecs = [model.get_vector(w) for w in words if w in model.key_to_index]
        if word_vecs:
            all_phrase_vectors.append(np.mean(word_vecs, axis=0))
    if not all_phrase_vectors:
        return None
    return np.mean(all_phrase_vectors, axis=0).tolist()

# Build semantic lexicons (centroids)
semantic_lexicons = {}
for source_key in user_text_lexicons:
    semantic_lexicons[source_key] = {}
    for persona in ["singles", "couples", "families", "remote_workers"]:
        semantic_lexicons[source_key][persona] = {
            "positive": get_phrase_vector(user_text_lexicons[source_key][persona]["positive"], w2v_model),
            "negative": get_phrase_vector(user_text_lexicons[source_key][persona]["negative"], w2v_model),
        }

semantic_lexicons["amenities"] = {}
for persona, a_list in full_config_data["amenities_map"].items():
    semantic_lexicons["amenities"][persona] = get_phrase_vector(a_list, w2v_model)

semantic_lexicons["highlights"] = {}
for persona, h_dict in full_config_data["highlights_map"].items():
    semantic_lexicons["highlights"][persona] = get_phrase_vector(list(h_dict.keys()), w2v_model)

full_config_data["semantic_lexicons"] = semantic_lexicons

# Broadcast config + model
bc_config = spark.sparkContext.broadcast(full_config_data)
bc_w2v_model = spark.sparkContext.broadcast(w2v_model)

# 2) PRECOMPUTE: TEXT to VECTOR UDFS
def text_to_vector_func(text):
    if text is None:
        return None
    s = str(text).strip()
    if not s:
        return None

    model = bc_w2v_model.value
    words = s.lower().replace("||", " ").replace(",", " ").split()
    vecs = [model.get_vector(w) for w in words if w in model.key_to_index]
    if not vecs:
        return None
    return np.mean(vecs, axis=0).tolist()

vector_udf = udf(text_to_vector_func, ArrayType(DoubleType()))

def average_vectors_func(vectors_list):
    if not vectors_list:
        return None
    valid = [v for v in vectors_list if v is not None]
    if not valid:
        return None
    return np.mean(valid, axis=0).tolist()

avg_vec_udf = udf(average_vectors_func, ArrayType(DoubleType()))

# 3) CLEAN PLACES + VECTORIZE
def clean_places_data_and_vectorize(df, city_name):
    current_columns = df.columns

    # latitude
    if "latitude" in current_columns:
        lat_col = "latitude"
    elif "lat" in current_columns:
        lat_col = "lat"
    else:
        raise ValueError(f"Missing latitude column in {city_name}. Found: {current_columns}")

    # longitude
    if "longitude" in current_columns:
        lon_col = "longitude"
    elif "long" in current_columns:
        lon_col = "long"
    else:
        raise ValueError(f"Missing longitude column in {city_name}. Found: {current_columns}")

    category_col = "category" if "category" in current_columns else current_columns[1]

    return (
        df.select(
            col("place_name"),
            col(category_col).alias("place_category"),
            col(lat_col).cast("double").alias("place_lat"),
            col(lon_col).cast("double").alias("place_lon"),
            col("reviews_content").alias("place_review"),
            lit(city_name).alias("source_city"),
        )
        .filter(col("place_lat").isNotNull() & col("place_lon").isNotNull())
        .withColumn("place_vec", vector_udf(col("place_review")))
    )

city_dfs = [(df_obj, city_key.replace("_", " ").title()) for city_key, df_obj in dfs.items()]
cleaned_dfs_list = [clean_places_data_and_vectorize(df, name) for df, name in city_dfs]
places_clean_df = reduce(DataFrame.unionByName, cleaned_dfs_list).cache()

print(f"Places loaded with vectors. Count: {places_clean_df.count()}")
display(places_clean_df.select("source_city").distinct())

# 4) PREPARE LISTINGS + VECTORIZE
listings_ready_df = (
    unique_listings
    .filter(col("lat").isNotNull() & col("long").isNotNull())
    .withColumn("highlights_vec", vector_udf(col("highlights_str")))
    .withColumn("desc_vec", vector_udf(col("description")))
    .withColumn("prop_reviews_vec", vector_udf(col("reviews_str")))
    .withColumn("amenities_vec", vector_udf(col("amenities_str")))
)

# 5) OPTIMIZED SPATIAL JOIN: GRID BUCKETING + NEIGHBORS + BBOX
CELL_SIZE_DEG = float(RADIUS_KM / 111.0)  # ~0.0135 deg for 1.5km

listings_grid = (
    listings_ready_df
    .withColumn("cell_x", floor(col("lat") / lit(CELL_SIZE_DEG)).cast("long"))
    .withColumn("cell_y", floor(col("long") / lit(CELL_SIZE_DEG)).cast("long"))
)

places_grid = (
    places_clean_df
    .withColumn("cell_x", floor(col("place_lat") / lit(CELL_SIZE_DEG)).cast("long"))
    .withColumn("cell_y", floor(col("place_lon") / lit(CELL_SIZE_DEG)).cast("long"))
)

neighbors = array(
    struct((col("cell_x") + lit(-1)).alias("nx"), (col("cell_y") + lit(-1)).alias("ny")),
    struct((col("cell_x") + lit(-1)).alias("nx"), (col("cell_y") + lit( 0)).alias("ny")),
    struct((col("cell_x") + lit(-1)).alias("nx"), (col("cell_y") + lit( 1)).alias("ny")),
    struct((col("cell_x") + lit( 0)).alias("nx"), (col("cell_y") + lit(-1)).alias("ny")),
    struct((col("cell_x") + lit( 0)).alias("nx"), (col("cell_y") + lit( 0)).alias("ny")),
    struct((col("cell_x") + lit( 0)).alias("nx"), (col("cell_y") + lit( 1)).alias("ny")),
    struct((col("cell_x") + lit( 1)).alias("nx"), (col("cell_y") + lit(-1)).alias("ny")),
    struct((col("cell_x") + lit( 1)).alias("nx"), (col("cell_y") + lit( 0)).alias("ny")),
    struct((col("cell_x") + lit( 1)).alias("nx"), (col("cell_y") + lit( 1)).alias("ny")),
)

listings_expanded = (
    listings_grid
    .withColumn("nbr", explode(neighbors))
    .withColumn("join_x", col("nbr.nx"))
    .withColumn("join_y", col("nbr.ny"))
    .drop("nbr")
)

candidates = (
    listings_expanded.join(
        places_grid,
        on=[
            listings_expanded["source_city"] == places_grid["source_city"],
            listings_expanded["join_x"] == places_grid["cell_x"],
            listings_expanded["join_y"] == places_grid["cell_y"],
        ],
        how="inner"
    ).drop(places_grid["source_city"])
)

lat_delta = lit(RADIUS_KM / 111.0)
lon_delta = lit(RADIUS_KM / 111.0) / cos(radians(col("lat")))

candidates_bbox = candidates.filter(
    (ps_abs(col("place_lat") - col("lat")) <= lat_delta) &
    (ps_abs(col("place_lon") - col("long")) <= lon_delta)
)

dist_expr = acos(
    sin(radians(col("lat"))) * sin(radians(col("place_lat"))) +
    cos(radians(col("lat"))) * cos(radians(col("place_lat"))) *
    cos(radians(col("place_lon")) - radians(col("long")))
) * lit(EARTH_KM)

nearby_places = (
    candidates_bbox
    .withColumn("distance", dist_expr)
    .filter(col("distance") <= lit(RADIUS_KM))
)

# 6) Aggeration: collect vectors + centroid for nearby context
nearby_stats = nearby_places.groupBy(
    "source_city", "listing_name", "url", "image", "lat", "long",
    "description", "guests", "ratings",
    "amenities_str", "highlights_str", "reviews_str",
    "highlights_vec", "desc_vec", "prop_reviews_vec", "amenities_vec"
).agg(
    count("place_name").alias("nearby_count"),
    collect_list("place_category").alias("nearby_categories_list"),
    collect_list("place_vec").alias("nearby_vecs_list"),
)

final_prep_df = (
    nearby_stats
    .withColumn("nearby_context_vec", avg_vec_udf(col("nearby_vecs_list")))
    .drop("nearby_vecs_list")
)

Places loaded with vectors. Count: 32475


source_city
Barcelona
Budapest
Buenos Aires
Lisbon
Mexico City
Miami
New York
Paris
Phuket
Tel Aviv


**Persona Scoring Engine and Final Output:**

In [0]:
# Persona Scoring + Final Output Table 
# Design Intent:
# - Singles & Couples should cluster toward "central" areas:
#   higher density (nearby_count), nightlife, dining, culture.
# - Remote Workers should prefer "farther from center":
#   lower density, quieter areas, but with work infrastructure + WiFi.
# - Families should have reduced impact (avoid dominating the dataset).
# - Explainability: debug JSON per persona.
# Decision Rules:
# (A) Scores are computed per persona (model output).
# (B) Dominant persona is chosen fairly using percentile calibration.
# (C) Presentation normalization: the dominant persona score is set to the max
# among the four displayed scores (for consistency in the final table),
# while raw scores are preserved in *_raw columns.

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, MapType, StructType, StructField
from pyspark.sql.window import Window

# Global configuration
PERSONAS = ["singles", "couples", "families", "remote_workers"]

THRESHOLD_HIGHLIGHTS = 0.80
THRESHOLD_AMENITIES  = 0.65

SIGMOID_SLOPE = 0.038

# Bias tuning:
# - Families bias increased to reduces dominance
# - Singles/Couples bias decreased to easier to stand out centrally
BIAS_MAP = {
    "singles": 56.0,
    "couples": 56.0,
    "families": 70.0,
    "remote_workers": 62.0,
}

OVERALL_MODE = "top2"  


# 7) SCORING UDF 
def calculate_scores_from_vectors_with_debug(
    h_vec, d_vec, p_vec, a_vec, n_vec,
    highlights_str, desc_str, amenities_str,
    nearby_cats, guests, nearby_count
):
    import numpy as np
    import math
    import json
    import re

    conf = bc_config.value
    sem_lex = conf["semantic_lexicons"]

    def cosine_sim(vec_a, vec_b):
        if vec_a is None or vec_b is None:
            return 0.0
        try:
            va = np.array(vec_a, dtype=float)
            vb = np.array(vec_b, dtype=float)
            na = np.linalg.norm(va)
            nb = np.linalg.norm(vb)
            if na == 0.0 or nb == 0.0:
                return 0.0
            return float(np.dot(va, vb) / (na * nb))
        except Exception:
            return 0.0

    def safe_int(x, default=0):
        try:
            return int(float(x)) if x is not None and x != "" else default
        except Exception:
            return default

    # Normalize input texts
    highlights_lower = str(highlights_str).lower() if highlights_str else ""
    desc_lower       = str(desc_str).lower()      if desc_str else ""
    amenities_lower  = str(amenities_str).lower() if amenities_str else ""

    # Robust nearby categories counting
    TARGET_CATS = [
        "nightlife",
        "culture",
        "wellness_lifestyle",
        "dining",
        "parks_recreation",
        "work_infrastructure",
    ]

    def parse_nearby_tokens(x):
        if isinstance(x, (list, tuple)):
            return [str(t).lower() for t in x if t is not None]
        s = str(x).lower() if x is not None else ""
        return re.findall(r"[a-z_]+", s)

    nearby_tokens = parse_nearby_tokens(nearby_cats)
    token_counts = {c: 0 for c in TARGET_CATS}
    for t in nearby_tokens:
        if t in token_counts:
            token_counts[t] += 1

    nightlife_count = token_counts["nightlife"]
    culture_hits    = token_counts["culture"]
    wellness_hits   = token_counts["wellness_lifestyle"]
    dining_hits     = token_counts["dining"]
    parks_count     = token_counts["parks_recreation"]
    work_hits       = token_counts["work_infrastructure"]

    # Flags
    is_quiet = ("quiet" in desc_lower) or ("residential" in desc_lower) or ("peaceful" in desc_lower)
    has_baby_gear = ("crib" in amenities_lower) or ("high chair" in amenities_lower) or ("baby" in amenities_lower)
    has_romantic_gear = ("jacuzzi" in amenities_lower) or ("hot tub" in amenities_lower) or ("sauna" in amenities_lower)
    has_wifi = ("wifi" in amenities_lower)

    guests_int       = safe_int(guests, 0)
    nearby_count_int = safe_int(nearby_count, 0)

    # Density proxy (centrality)
    density_base = math.log1p(max(nearby_count_int, 0))

    scores = {}
    debug_strings = {}

    for persona in PERSONAS:
        raw_score = 0.0

        # (1) Semantic scoring (POS - NEG)
        sem_desc_pos = cosine_sim(d_vec, sem_lex["listing_description"][persona]["positive"]) * 14.0
        sem_prop_pos = cosine_sim(p_vec, sem_lex["property_reviews"][persona]["positive"])    * 18.0
        sem_near_pos = cosine_sim(n_vec, sem_lex["nearby_places_reviews"][persona]["positive"]) * 12.0
        sem_pos = sem_desc_pos + sem_prop_pos + sem_near_pos

        sem_prop_neg = cosine_sim(p_vec, sem_lex["property_reviews"][persona]["negative"])   * 22.0
        sem_desc_neg = cosine_sim(d_vec, sem_lex["listing_description"][persona]["negative"])* 12.0
        sem_near_neg = cosine_sim(n_vec, sem_lex["nearby_places_reviews"][persona]["negative"]) * 16.0
        sem_neg = sem_prop_neg + sem_desc_neg + sem_near_neg

        raw_score += sem_pos
        raw_score -= sem_neg

        # (2) Vector boosts (threshold-gated)
        sim_h = cosine_sim(h_vec, sem_lex["highlights"][persona])
        highlight_vec_boost = (sim_h * 12.0) if (sim_h > THRESHOLD_HIGHLIGHTS) else 0.0
        raw_score += highlight_vec_boost

        sim_a = cosine_sim(a_vec, sem_lex["amenities"][persona])
        amenities_vec_boost = (sim_a * 12.0) if (sim_a > THRESHOLD_AMENITIES) else 0.0
        raw_score += amenities_vec_boost

        # (3) Keyword boosts (highlights substring match)
        kw_boost = 0.0
        hits_list = []
        persona_h_map = conf["highlights_map"].get(persona, {})
        for h_text, bonus in persona_h_map.items():
            if h_text and (h_text.lower() in highlights_lower):
                mult = 3.0 if persona == "remote_workers" else 1.4
                added = float(bonus) * mult
                kw_boost += added
                hits_list.append(f"{h_text}:{round(added, 2)}")

        raw_score += kw_boost

        # (4) Rule-based logic (Center vs. Farther-from-center)
        logic_score = 0.0

        if persona in ["singles", "couples"]:
            logic_score += density_base * 3.2
        elif persona == "remote_workers":
            logic_score -= density_base * 2.8
        else:
            logic_score += density_base * 0.6

        if persona == "singles":
            logic_score += min(nightlife_count, 20) * 1.7
            logic_score += min(dining_hits, 20) * 1.1
            logic_score += min(culture_hits, 14) * 1.0
            if is_quiet:
                logic_score -= 10.0
            if guests_int >= 4:
                logic_score -= 6.0

        elif persona == "couples":
            if has_romantic_gear:
                logic_score += 16.0
            logic_score += min(dining_hits, 18) * 1.2
            logic_score += min(wellness_hits, 14) * 1.8
            logic_score += min(culture_hits, 14) * 1.5
            if nightlife_count > 16:
                logic_score -= 6.0
            if is_quiet:
                logic_score += 4.0

        elif persona == "families":
            if guests_int < 3:
                logic_score -= 10.0
            elif guests_int == 3:
                logic_score += 4.0
            elif guests_int >= 4:
                logic_score += 10.0

            if has_baby_gear:
                logic_score += 10.0

            logic_score += min(parks_count, 18) * 2.6
            logic_score -= min(nightlife_count, 25) * 1.2

            if is_quiet:
                logic_score += 6.0

        elif persona == "remote_workers":
            if not has_wifi:
                logic_score -= 28.0
            else:
                if "workspace" in desc_lower or "desk" in desc_lower: #stonger wheight
                    logic_score += 10.0
                if "fast" in desc_lower:
                    logic_score += 6.0

            logic_score += min(work_hits, 14) * 4.6

            if is_quiet:
                logic_score += 10.0
            if nightlife_count > 10:
                logic_score -= 8.0
            if guests_int >= 5:
                logic_score -= 6.0

        raw_score += logic_score

        # (5) Normaliziation
        bias = float(BIAS_MAP.get(persona, 58.0))
        norm_score = 100.0 / (1.0 + math.exp(-SIGMOID_SLOPE * (raw_score - bias)))
        norm_score = float(round(norm_score, 1))
        scores[persona] = norm_score

        # Debug 
        dbg = {
            "sem_pos_total": round(sem_pos, 2),
            "sem_neg_total": round(-sem_neg, 2),
            "highlight_vec_sim": round(sim_h, 3),
            "highlight_vec_boost": round(highlight_vec_boost, 2),
            "amenities_vec_sim": round(sim_a, 3),
            "amenities_vec_boost": round(amenities_vec_boost, 2),
            "keyword_boost_total": round(kw_boost, 2),
            "keyword_hits": hits_list[:20],
            "logic_score": round(logic_score, 2),
            "density_base_log1p": round(density_base, 3),
            "nearby_counts": {
                "nightlife": int(nightlife_count),
                "culture": int(culture_hits),
                "wellness": int(wellness_hits),
                "dining": int(dining_hits),
                "parks": int(parks_count),
                "work_infra": int(work_hits),
                "nearby_count": int(nearby_count_int),
            },
            "flags": {
                "is_quiet": bool(is_quiet),
                "has_wifi": bool(has_wifi),
                "has_baby_gear": bool(has_baby_gear),
                "has_romantic_gear": bool(has_romantic_gear),
            },
            "raw_score": round(raw_score, 2),
            "bias": float(bias),
            "sigmoid_slope": float(SIGMOID_SLOPE),
            "norm_score": float(norm_score),
        }

        debug_strings[persona] = json.dumps(dbg, ensure_ascii=False)

    return (scores, debug_strings)


# UDF schema: (scores_map, debug_map)
score_debug_schema = StructType([
    StructField("scores", MapType(StringType(), DoubleType()), True),
    StructField("debug",  MapType(StringType(), StringType()), True),
])

score_vectors_debug_udf = F.udf(calculate_scores_from_vectors_with_debug, score_debug_schema)


# 8) Execute + Final table 
final_prep_df = final_prep_df.repartition(spark.sparkContext.defaultParallelism)

result_df = (
    final_prep_df
    .withColumn(
        "scoring",
        score_vectors_debug_udf(
            F.col("highlights_vec"),
            F.col("desc_vec"),
            F.col("prop_reviews_vec"),
            F.col("amenities_vec"),
            F.col("nearby_context_vec"),
            F.col("highlights_str"),
            F.col("description"),
            F.col("amenities_str"),
            F.col("nearby_categories_list"),
            F.col("guests"),
            F.col("nearby_count"),
        )
    )
    .withColumn("scores", F.col("scoring.scores"))
    .withColumn("debug",  F.col("scoring.debug"))
)

final_display = (
    result_df.select(
        F.col("source_city"),
        F.col("listing_name"),
        F.col("image"),
        F.col("url"),
        F.col("guests"),
        F.col("nearby_count"),
        F.col("ratings").alias("rating"),

        F.coalesce(F.col("scores.singles"),        F.lit(0.0)).alias("score_singles"),
        F.coalesce(F.col("scores.couples"),        F.lit(0.0)).alias("score_couples"),
        F.coalesce(F.col("scores.families"),       F.lit(0.0)).alias("score_families"),
        F.coalesce(F.col("scores.remote_workers"), F.lit(0.0)).alias("score_work"),

        F.col("highlights_str"),

        F.col("debug.singles").alias("debug_singles"),
        F.col("debug.couples").alias("debug_couples"),
        F.col("debug.families").alias("debug_families"),
        F.col("debug.remote_workers").alias("debug_work"),

        F.col("lat"),
        F.col("long"),
    )
)

# 9) Dominant persona (Percentile-based decision rule)
w_s  = Window.orderBy(F.col("score_singles"))
w_c  = Window.orderBy(F.col("score_couples"))
w_f  = Window.orderBy(F.col("score_families"))
w_rw = Window.orderBy(F.col("score_work"))

final_display = (
    final_display
    .withColumn("pct_singles",  F.percent_rank().over(w_s))
    .withColumn("pct_couples",  F.percent_rank().over(w_c))
    .withColumn("pct_families", F.percent_rank().over(w_f))
    .withColumn("pct_work",     F.percent_rank().over(w_rw))
    .withColumn(
        "persona_rank",
        F.array(
            F.struct(F.col("pct_singles").alias("score"),  F.lit("Singles").alias("persona")),
            F.struct(F.col("pct_couples").alias("score"),  F.lit("Couples").alias("persona")),
            F.struct(F.col("pct_families").alias("score"), F.lit("Families").alias("persona")),
            F.struct(F.col("pct_work").alias("score"),     F.lit("Remote Workers").alias("persona")),
        )
    )
    .withColumn("best_persona_struct", F.array_max(F.col("persona_rank")))
    .withColumn("dominant_persona", F.col("best_persona_struct.persona"))
    .drop("persona_rank", "best_persona_struct")
)

# 9.1) Preserve RAW scores (for evaluation/debug), then normalize display:
# Ensure the dominant persona score is the highest among the four (presentation).
from pyspark.sql import functions as F

final_display = (
    final_display

    # Preserve original (pre-presentation) scores
    .withColumn("score_singles_raw",  F.col("score_singles"))
    .withColumn("score_couples_raw",  F.col("score_couples"))
    .withColumn("score_families_raw", F.col("score_families"))
    .withColumn("score_work_raw",     F.col("score_work"))

    # Compute max score across personas (normalized scores)
    .withColumn(
        "max_score",
        F.greatest(
            F.col("score_singles"),
            F.col("score_couples"),
            F.col("score_families"),
            F.col("score_work")
        )
    )

    # For presentation only:
    # Force the dominant persona to display the max score
    # Dominant persona already incorporates percentile + raw tie-break
    .withColumn(
        "score_singles",
        F.when(F.col("dominant_persona") == F.lit("Singles"), F.col("max_score"))
         .otherwise(F.col("score_singles"))
    )
    .withColumn(
        "score_couples",
        F.when(F.col("dominant_persona") == F.lit("Couples"), F.col("max_score"))
         .otherwise(F.col("score_couples"))
    )
    .withColumn(
        "score_families",
        F.when(F.col("dominant_persona") == F.lit("Families"), F.col("max_score"))
         .otherwise(F.col("score_families"))
    )
    .withColumn(
        "score_work",
        F.when(F.col("dominant_persona") == F.lit("Remote Workers"), F.col("max_score"))
         .otherwise(F.col("score_work"))
    )

    .drop("max_score")
)


# 10) Overall score
# Important: Use RAW scores for overall_score to avoid inflating ranking due to display normalization.
if OVERALL_MODE == "top2":
    final_display = (
        final_display
        .withColumn(
            "sorted_scores_desc_raw",
            F.expr(
                "array_sort(array(score_singles_raw, score_couples_raw, score_families_raw, score_work_raw), "
                "(l, r) -> case when l > r then -1 when l < r then 1 else 0 end)"
            )
        )
        .withColumn(
            "overall_score",
            (F.col("sorted_scores_desc_raw")[0] + F.col("sorted_scores_desc_raw")[1]) / F.lit(2.0)
        )
        .drop("sorted_scores_desc_raw")
    )
else:
    final_display = final_display.withColumn(
        "overall_score",
        (F.col("score_singles_raw") + F.col("score_couples_raw") + F.col("score_families_raw") + F.col("score_work_raw")) / F.lit(4.0)
    )

final_display.cache()
print(f"Deep Analysis (With Debug) Complete. Final Count: {final_display.count()}")

display(final_display.orderBy(F.col('overall_score').desc()).limit(100))

# Distribution check
final_display.groupBy("dominant_persona").count().display()

# Separation check (RAW separation is more honest for evaluation)
final_display.selectExpr(
  "greatest(score_singles_raw,score_couples_raw,score_families_raw,score_work_raw) - "
  "least(score_singles_raw,score_couples_raw,score_families_raw,score_work_raw) as persona_gap_raw"
).summary().display()

# Cleanup 
try:
    places_clean_df.unpersist()
except Exception:
    pass

try:
    result_df.unpersist()
except Exception:
    pass

Deep Analysis (With Debug) Complete. Final Count: 14363


source_city,listing_name,image,url,guests,nearby_count,rating,score_singles,score_couples,score_families,score_work,highlights_str,debug_singles,debug_couples,debug_families,debug_work,lat,long,pct_singles,pct_couples,pct_families,pct_work,dominant_persona,score_singles_raw,score_couples_raw,score_families_raw,score_work_raw,overall_score
Budapest,"Entire apartment in Budapest, Hungary",https://a0.muscache.com/pictures/e7f02b21-eac4-4555-bcc3-db181a97c426.jpg,https://www.airbnb.com/rooms/53903632,2,1157,4.87,91.5,87.8,14.3,38.7,"Self check-in, Vibrant neighbourhood, Timea is a Superhost","{""sem_pos_total"": 39.44, ""sem_neg_total"": -47.13, ""highlight_vec_sim"": 0.881, ""highlight_vec_boost"": 10.57, ""amenities_vec_sim"": 0.654, ""amenities_vec_boost"": 7.84, ""keyword_boost_total"": 15.4, ""keyword_hits"": [""self check-in:7.0"", ""vibrant neighbourhood:5.6"", ""superhost:2.8""], ""logic_score"": 92.57, ""density_base_log1p"": 7.054, ""nearby_counts"": {""nightlife"": 306, ""culture"": 173, ""wellness"": 168, ""dining"": 275, ""parks"": 128, ""work_infra"": 106, ""nearby_count"": 1157}, ""flags"": {""is_quiet"": false, ""has_wifi"": true, ""has_baby_gear"": false, ""has_romantic_gear"": true}, ""raw_score"": 118.69, ""bias"": 56.0, ""sigmoid_slope"": 0.038, ""norm_score"": 91.5}","{""sem_pos_total"": 36.65, ""sem_neg_total"": -45.78, ""highlight_vec_sim"": 0.708, ""highlight_vec_boost"": 0.0, ""amenities_vec_sim"": 0.816, ""amenities_vec_boost"": 9.79, ""keyword_boost_total"": 7.0, ""keyword_hits"": [""superhost:4.2"", ""self check-in:2.8""], ""logic_score"": 100.37, ""density_base_log1p"": 7.054, ""nearby_counts"": {""nightlife"": 306, ""culture"": 173, ""wellness"": 168, ""dining"": 275, ""parks"": 128, ""work_infra"": 106, ""nearby_count"": 1157}, ""flags"": {""is_quiet"": false, ""has_wifi"": true, ""has_baby_gear"": false, ""has_romantic_gear"": true}, ""raw_score"": 108.04, ""bias"": 56.0, ""sigmoid_slope"": 0.038, ""norm_score"": 87.8}","{""sem_pos_total"": 39.23, ""sem_neg_total"": -45.86, ""highlight_vec_sim"": 0.707, ""highlight_vec_boost"": 0.0, ""amenities_vec_sim"": 0.836, ""amenities_vec_boost"": 10.03, ""keyword_boost_total"": 8.4, ""keyword_hits"": [""superhost:5.6"", ""self check-in:2.8""], ""logic_score"": 11.03, ""density_base_log1p"": 7.054, ""nearby_counts"": {""nightlife"": 306, ""culture"": 173, ""wellness"": 168, ""dining"": 275, ""parks"": 128, ""work_infra"": 106, ""nearby_count"": 1157}, ""flags"": {""is_quiet"": false, ""has_wifi"": true, ""has_baby_gear"": false, ""has_romantic_gear"": true}, ""raw_score"": 22.84, ""bias"": 70.0, ""sigmoid_slope"": 0.038, ""norm_score"": 14.3}","{""sem_pos_total"": 38.29, ""sem_neg_total"": -46.0, ""highlight_vec_sim"": 0.701, ""highlight_vec_boost"": 0.0, ""amenities_vec_sim"": 0.618, ""amenities_vec_boost"": 0.0, ""keyword_boost_total"": 21.0, ""keyword_hits"": [""self check-in:15.0"", ""superhost:6.0""], ""logic_score"": 36.65, ""density_base_log1p"": 7.054, ""nearby_counts"": {""nightlife"": 306, ""culture"": 173, ""wellness"": 168, ""dining"": 275, ""parks"": 128, ""work_infra"": 106, ""nearby_count"": 1157}, ""flags"": {""is_quiet"": false, ""has_wifi"": true, ""has_baby_gear"": false, ""has_romantic_gear"": true}, ""raw_score"": 49.94, ""bias"": 62.0, ""sigmoid_slope"": 0.038, ""norm_score"": 38.7}",47.49559,19.0546,0.9997911154435315,0.9979111544353154,0.27475281994151235,0.7265004873972984,Singles,91.5,87.8,14.3,38.7,89.65
Paris,"Room in boutique hotel in Paris, France",https://a0.muscache.com/pictures/miso/Hosting-38900228/original/c1026e1b-1675-4362-980e-c699f39b5e05.jpeg,https://www.airbnb.com/rooms/38900228?adults=2&children=0&infants=0&pets=0,2,822,4.77,89.1,89.1,12.9,32.1,"Exceptional check-in experience, Great location, ⁨Hôtel 34B***⁩ is a Superhost","{""sem_pos_total"": 40.29, ""sem_neg_total"": -47.6, ""highlight_vec_sim"": 0.869, ""highlight_vec_boost"": 10.43, ""amenities_vec_sim"": 0.684, ""amenities_vec_boost"": 8.21, ""keyword_boost_total

dominant_persona,count
Couples,2536
Singles,3657
Families,4834
Remote Workers,3336


summary,persona_gap_raw
count,14363
mean,48.78307456659467
stddev,19.33648278757763
min,1.6000000000000014
25%,38.8
50%,54.0
75%,63.300000000000004
max,83.1


**Internal-Only personas model rating:**

In [0]:
# Cell X - Internal Persona Score Only + Final Output Table 
# Final Version (Internal Data Only) - Same Output Columns

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, MapType, StructType, StructField, ArrayType
from pyspark.sql.window import Window

# Global configuration
PERSONAS = ["singles", "couples", "families", "remote_workers"]

THRESHOLD_HIGHLIGHTS = 0.80
THRESHOLD_AMENITIES  = 0.65

SIGMOID_SLOPE = 0.038

BIAS_MAP = {
    "singles": 56.0,
    "couples": 56.0,
    "families": 70.0,
    "remote_workers": 62.0,
}

OVERALL_MODE = "top2"  # "top2" or "mean4"

# 0) Build internal-only input DF (keep same required columns)
# Assumption: final_prep_df already exists with Internal vectors & fields.
# We create placeholders for external data columns but keep names identical.

final_prep_df_internal = (
    final_prep_df
    .select(
        # keep original identifiers/metadata
        F.col("source_city"),
        F.col("listing_name"),
        F.col("image"),
        F.col("url"),
        F.col("guests"),
        F.col("ratings"),
        F.col("lat"),
        F.col("long"),

        # Internal Text fields
        F.col("highlights_str"),
        F.col("description"),
        F.col("amenities_str"),

        # Internal vectors (NLP outputs)
        F.col("highlights_vec"),
        F.col("desc_vec"),
        F.col("prop_reviews_vec"),
        F.col("amenities_vec"),

        # Extnral placeholders (to keep identical schema)
        F.lit(0).cast("int").alias("nearby_count"),
        F.array().cast(ArrayType(StringType())).alias("nearby_categories_list"),

        # For scoring: we still need nearby_context_vec argument in the UDF.
        # We set it to Internal context vector:
        # prefer prop_reviews_vec; fallback to desc_vec if prop is null.
        F.coalesce(F.col("prop_reviews_vec"), F.col("desc_vec")).alias("nearby_context_vec"),
    )
)

final_prep_df_internal = final_prep_df_internal.repartition(spark.sparkContext.defaultParallelism)

# 1) Scoring UDF (same logic, with explainable debug)
def calculate_scores_from_vectors_with_debug(
    h_vec, d_vec, p_vec, a_vec, n_vec,
    highlights_str, desc_str, amenities_str,
    nearby_cats, guests, nearby_count
):
    import numpy as np
    import math
    import json
    import re

    conf = bc_config.value
    sem_lex = conf["semantic_lexicons"]

    def cosine_sim(vec_a, vec_b):
        if vec_a is None or vec_b is None:
            return 0.0
        try:
            va = np.array(vec_a, dtype=float)
            vb = np.array(vec_b, dtype=float)
            na = np.linalg.norm(va)
            nb = np.linalg.norm(vb)
            if na == 0.0 or nb == 0.0:
                return 0.0
            return float(np.dot(va, vb) / (na * nb))
        except Exception:
            return 0.0

    def safe_int(x, default=0):
        try:
            return int(float(x)) if x is not None and x != "" else default
        except Exception:
            return default

    # Normalize input texts
    highlights_lower = str(highlights_str).lower() if highlights_str else ""
    desc_lower       = str(desc_str).lower()      if desc_str else ""
    amenities_lower  = str(amenities_str).lower() if amenities_str else ""

    # Robust nearby categories counting (in internal-only mode it will be empty / 0)
    TARGET_CATS = [
        "nightlife",
        "culture",
        "wellness_lifestyle",
        "dining",
        "parks_recreation",
        "work_infrastructure",
    ]

    def parse_nearby_tokens(x):
        if isinstance(x, (list, tuple)):
            return [str(t).lower() for t in x if t is not None]
        s = str(x).lower() if x is not None else ""
        return re.findall(r"[a-z_]+", s)

    nearby_tokens = parse_nearby_tokens(nearby_cats)
    token_counts = {c: 0 for c in TARGET_CATS}
    for t in nearby_tokens:
        if t in token_counts:
            token_counts[t] += 1

    nightlife_count = token_counts["nightlife"]
    culture_hits    = token_counts["culture"]
    wellness_hits   = token_counts["wellness_lifestyle"]
    dining_hits     = token_counts["dining"]
    parks_count     = token_counts["parks_recreation"]
    work_hits       = token_counts["work_infrastructure"]

    # Flags
    is_quiet = ("quiet" in desc_lower) or ("residential" in desc_lower) or ("peaceful" in desc_lower)
    has_baby_gear = ("crib" in amenities_lower) or ("high chair" in amenities_lower) or ("baby" in amenities_lower)
    has_romantic_gear = ("jacuzzi" in amenities_lower) or ("hot tub" in amenities_lower) or ("sauna" in amenities_lower)
    has_wifi = ("wifi" in amenities_lower)

    guests_int       = safe_int(guests, 0)
    nearby_count_int = safe_int(nearby_count, 0)

    # Density proxy (centrality) — internal-only to nearby_count=0 by default
    density_base = math.log1p(max(nearby_count_int, 0))

    scores = {}
    debug_strings = {}

    for persona in PERSONAS:
        raw_score = 0.0

        # (1) Semantic scoring (POS - NEG)
        sem_desc_pos = cosine_sim(d_vec, sem_lex["listing_description"][persona]["positive"]) * 14.0
        sem_prop_pos = cosine_sim(p_vec, sem_lex["property_reviews"][persona]["positive"])    * 18.0
        # n_vec is fed by Internal context vector in internal-only mode
        sem_near_pos = cosine_sim(n_vec, sem_lex["nearby_places_reviews"][persona]["positive"]) * 12.0
        sem_pos = sem_desc_pos + sem_prop_pos + sem_near_pos

        sem_prop_neg = cosine_sim(p_vec, sem_lex["property_reviews"][persona]["negative"])   * 22.0
        sem_desc_neg = cosine_sim(d_vec, sem_lex["listing_description"][persona]["negative"])* 12.0
        sem_near_neg = cosine_sim(n_vec, sem_lex["nearby_places_reviews"][persona]["negative"]) * 16.0
        sem_neg = sem_prop_neg + sem_desc_neg + sem_near_neg

        raw_score += sem_pos
        raw_score -= sem_neg

        # (2) Vector boosts (threshold-gated)
        sim_h = cosine_sim(h_vec, sem_lex["highlights"][persona])
        highlight_vec_boost = (sim_h * 12.0) if (sim_h > THRESHOLD_HIGHLIGHTS) else 0.0
        raw_score += highlight_vec_boost

        sim_a = cosine_sim(a_vec, sem_lex["amenities"][persona])
        amenities_vec_boost = (sim_a * 12.0) if (sim_a > THRESHOLD_AMENITIES) else 0.0
        raw_score += amenities_vec_boost

        # (3) Keyword boosts (highlights substring match)
        kw_boost = 0.0
        hits_list = []
        persona_h_map = conf["highlights_map"].get(persona, {})
        for h_text, bonus in persona_h_map.items():
            if h_text and (h_text.lower() in highlights_lower):
                mult = 3.0 if persona == "remote_workers" else 1.4
                added = float(bonus) * mult
                kw_boost += added
                hits_list.append(f"{h_text}:{round(added, 2)}")

        raw_score += kw_boost

        # (4) Rule-based logic (Center vs. Farther-from-center)
        logic_score = 0.0

        if persona in ["singles", "couples"]:
            logic_score += density_base * 3.2
        elif persona == "remote_workers":
            logic_score -= density_base * 2.8
        else:
            logic_score += density_base * 0.6

        if persona == "singles":
            logic_score += min(nightlife_count, 20) * 1.7
            logic_score += min(dining_hits, 20) * 1.1
            logic_score += min(culture_hits, 14) * 1.0
            if is_quiet:
                logic_score -= 10.0
            if guests_int >= 4:
                logic_score -= 6.0

        elif persona == "couples":
            if has_romantic_gear:
                logic_score += 16.0
            logic_score += min(dining_hits, 18) * 1.2
            logic_score += min(wellness_hits, 14) * 1.8
            logic_score += min(culture_hits, 14) * 1.5
            if nightlife_count > 16:
                logic_score -= 6.0
            if is_quiet:
                logic_score += 4.0

        elif persona == "families":
            if guests_int < 3:
                logic_score -= 10.0
            elif guests_int == 3:
                logic_score += 4.0
            elif guests_int >= 4:
                logic_score += 10.0

            if has_baby_gear:
                logic_score += 10.0

            logic_score += min(parks_count, 18) * 2.6
            logic_score -= min(nightlife_count, 25) * 1.2

            if is_quiet:
                logic_score += 6.0

        elif persona == "remote_workers":
            if not has_wifi:
                logic_score -= 28.0
            else:
                if "workspace" in desc_lower or "desk" in desc_lower:
                    logic_score += 10.0
                if "fast" in desc_lower:
                    logic_score += 6.0

            logic_score += min(work_hits, 14) * 4.6

            if is_quiet:
                logic_score += 10.0
            if nightlife_count > 10:
                logic_score -= 8.0
            if guests_int >= 5:
                logic_score -= 6.0

        raw_score += logic_score

        # (5) Normalization 
        bias = float(BIAS_MAP.get(persona, 58.0))
        norm_score = 100.0 / (1.0 + math.exp(-SIGMOID_SLOPE * (raw_score - bias)))
        norm_score = float(round(norm_score, 1))
        scores[persona] = norm_score

        # Debug JSON
        dbg = {
            "sem_pos_total": round(sem_pos, 2),
            "sem_neg_total": round(-sem_neg, 2),
            "highlight_vec_sim": round(sim_h, 3),
            "highlight_vec_boost": round(highlight_vec_boost, 2),
            "amenities_vec_sim": round(sim_a, 3),
            "amenities_vec_boost": round(amenities_vec_boost, 2),
            "keyword_boost_total": round(kw_boost, 2),
            "keyword_hits": hits_list[:20],
            "logic_score": round(logic_score, 2),
            "density_base_log1p": round(density_base, 3),
            "nearby_counts": {
                "nightlife": int(nightlife_count),
                "culture": int(culture_hits),
                "wellness": int(wellness_hits),
                "dining": int(dining_hits),
                "parks": int(parks_count),
                "work_infra": int(work_hits),
                "nearby_count": int(nearby_count_int),
            },
            "flags": {
                "is_quiet": bool(is_quiet),
                "has_wifi": bool(has_wifi),
                "has_baby_gear": bool(has_baby_gear),
                "has_romantic_gear": bool(has_romantic_gear),
            },
            "raw_score": round(raw_score, 2),
            "bias": float(bias),
            "sigmoid_slope": float(SIGMOID_SLOPE),
            "norm_score": float(norm_score),
        }

        debug_strings[persona] = json.dumps(dbg, ensure_ascii=False)

    return (scores, debug_strings)


score_debug_schema = StructType([
    StructField("scores", MapType(StringType(), DoubleType()), True),
    StructField("debug",  MapType(StringType(), StringType()), True),
])

score_vectors_debug_udf = F.udf(calculate_scores_from_vectors_with_debug, score_debug_schema)

# 2) Execute + Final table (same columns)
result_df_internal = (
    final_prep_df_internal
    .withColumn(
        "scoring",
        score_vectors_debug_udf(
            F.col("highlights_vec"),
            F.col("desc_vec"),
            F.col("prop_reviews_vec"),
            F.col("amenities_vec"),
            F.col("nearby_context_vec"),
            F.col("highlights_str"),
            F.col("description"),
            F.col("amenities_str"),
            F.col("nearby_categories_list"),
            F.col("guests"),
            F.col("nearby_count"),
        )
    )
    .withColumn("scores", F.col("scoring.scores"))
    .withColumn("debug",  F.col("scoring.debug"))
)

final_display_internal = (
    result_df_internal.select(
        F.col("source_city"),
        F.col("listing_name"),
        F.col("image"),
        F.col("url"),
        F.col("guests"),
        F.col("nearby_count"),
        F.col("ratings").alias("rating"),

        F.coalesce(F.col("scores.singles"),        F.lit(0.0)).alias("score_singles"),
        F.coalesce(F.col("scores.couples"),        F.lit(0.0)).alias("score_couples"),
        F.coalesce(F.col("scores.families"),       F.lit(0.0)).alias("score_families"),
        F.coalesce(F.col("scores.remote_workers"), F.lit(0.0)).alias("score_work"),

        F.col("highlights_str"),

        F.col("debug.singles").alias("debug_singles"),
        F.col("debug.couples").alias("debug_couples"),
        F.col("debug.families").alias("debug_families"),
        F.col("debug.remote_workers").alias("debug_work"),

        F.col("lat"),
        F.col("long"),
    )
)

# 3) Fair Dominant Persona (Percentile-based decision rule)
w_s  = Window.orderBy(F.col("score_singles"))
w_c  = Window.orderBy(F.col("score_couples"))
w_f  = Window.orderBy(F.col("score_families"))
w_rw = Window.orderBy(F.col("score_work"))

final_display_internal = (
    final_display_internal
    .withColumn("pct_singles",  F.percent_rank().over(w_s))
    .withColumn("pct_couples",  F.percent_rank().over(w_c))
    .withColumn("pct_families", F.percent_rank().over(w_f))
    .withColumn("pct_work",     F.percent_rank().over(w_rw))
    .withColumn(
        "persona_rank",
        F.array(
            F.struct(F.col("pct_singles").alias("score"),  F.lit("Singles").alias("persona")),
            F.struct(F.col("pct_couples").alias("score"),  F.lit("Couples").alias("persona")),
            F.struct(F.col("pct_families").alias("score"), F.lit("Families").alias("persona")),
            F.struct(F.col("pct_work").alias("score"),     F.lit("Remote Workers").alias("persona")),
        )
    )
    .withColumn("best_persona_struct", F.array_max(F.col("persona_rank")))
    .withColumn("dominant_persona", F.col("best_persona_struct.persona"))
    .drop("persona_rank", "best_persona_struct")
)

# 4) Preserve RAW scores + presentation normalization (same as original)
final_display_internal = (
    final_display_internal
    .withColumn("score_singles_raw",  F.col("score_singles"))
    .withColumn("score_couples_raw",  F.col("score_couples"))
    .withColumn("score_families_raw", F.col("score_families"))
    .withColumn("score_work_raw",     F.col("score_work"))
    .withColumn(
        "max_score",
        F.greatest(
            F.col("score_singles"),
            F.col("score_couples"),
            F.col("score_families"),
            F.col("score_work"),
        )
    )
    .withColumn(
        "score_singles",
        F.when(F.col("dominant_persona") == F.lit("Singles"), F.col("max_score")).otherwise(F.col("score_singles"))
    )
    .withColumn(
        "score_couples",
        F.when(F.col("dominant_persona") == F.lit("Couples"), F.col("max_score")).otherwise(F.col("score_couples"))
    )
    .withColumn(
        "score_families",
        F.when(F.col("dominant_persona") == F.lit("Families"), F.col("max_score")).otherwise(F.col("score_families"))
    )
    .withColumn(
        "score_work",
        F.when(F.col("dominant_persona") == F.lit("Remote Workers"), F.col("max_score")).otherwise(F.col("score_work"))
    )
    .drop("max_score")
)

# 5) Overall score (RAW only)
if OVERALL_MODE == "top2":
    final_display_internal = (
        final_display_internal
        .withColumn(
            "sorted_scores_desc_raw",
            F.expr(
                "array_sort(array(score_singles_raw, score_couples_raw, score_families_raw, score_work_raw), "
                "(l, r) -> case when l > r then -1 when l < r then 1 else 0 end)"
            )
        )
        .withColumn(
            "overall_score",
            (F.col("sorted_scores_desc_raw")[0] + F.col("sorted_scores_desc_raw")[1]) / F.lit(2.0)
        )
        .drop("sorted_scores_desc_raw")
    )
else:
    final_display_internal = final_display_internal.withColumn(
        "overall_score",
        (F.col("score_singles_raw") + F.col("score_couples_raw") + F.col("score_families_raw") + F.col("score_work_raw")) / F.lit(4.0)
    )

final_display_internal.cache()
print(f"INTERNAL-ONLY Deep Analysis (With Debug) Complete. Final Count: {final_display_internal.count()}")

display(final_display_internal.orderBy(F.col("overall_score").desc()).limit(100))

# Distribution check
final_display_internal.groupBy("dominant_persona").count().display()

# Separation check (RAW separation is more honest for evaluation)
final_display_internal.selectExpr(
  "greatest(score_singles_raw,score_couples_raw,score_families_raw,score_work_raw) - "
  "least(score_singles_raw,score_couples_raw,score_families_raw,score_work_raw) as persona_gap_raw"
).summary().display()

INTERNAL-ONLY Deep Analysis (With Debug) Complete. Final Count: 14363


source_city,listing_name,image,url,guests,nearby_count,rating,score_singles,score_couples,score_families,score_work,highlights_str,debug_singles,debug_couples,debug_families,debug_work,lat,long,pct_singles,pct_couples,pct_families,pct_work,dominant_persona,score_singles_raw,score_couples_raw,score_families_raw,score_work_raw,overall_score
Miami,"Entire serviced apartment in Doral, Florida, United States",https://a0.muscache.com/pictures/6f5c7137-5667-45cd-8799-3380cbec4ee9.jpg,https://www.airbnb.ca/rooms/53987438,2,0,4.92,15.6,37.0,12.4,37.0,"Self check-in, Beautiful area, Maria is a Superhost","{""sem_pos_total"": 40.59, ""sem_neg_total"": -47.62, ""highlight_vec_sim"": 0.886, ""highlight_vec_boost"": 10.63, ""amenities_vec_sim"": 0.686, ""amenities_vec_boost"": 8.23, ""keyword_boost_total"": 9.8, ""keyword_hits"": [""self check-in:7.0"", ""superhost:2.8""], ""logic_score"": -10.0, ""density_base_log1p"": 0.0, ""nearby_counts"": {""nightlife"": 0, ""culture"": 0, ""wellness"": 0, ""dining"": 0, ""parks"": 0, ""work_infra"": 0, ""nearby_count"": 0}, ""flags"": {""is_quiet"": true, ""has_wifi"": true, ""has_baby_gear"": true, ""has_romantic_gear"": true}, ""raw_score"": 11.63, ""bias"": 56.0, ""sigmoid_slope"": 0.038, ""norm_score"": 15.6}","{""sem_pos_total"": 38.02, ""sem_neg_total"": -46.59, ""highlight_vec_sim"": 0.804, ""highlight_vec_boost"": 9.65, ""amenities_vec_sim"": 0.834, ""amenities_vec_boost"": 10.0, ""keyword_boost_total"": 7.0, ""keyword_hits"": [""superhost:4.2"", ""self check-in:2.8""], ""logic_score"": 20.0, ""density_base_log1p"": 0.0, ""nearby_counts"": {""nightlife"": 0, ""culture"": 0, ""wellness"": 0, ""dining"": 0, ""parks"": 0, ""work_infra"": 0, ""nearby_count"": 0}, ""flags"": {""is_quiet"": true, ""has_wifi"": true, ""has_baby_gear"": true, ""has_romantic_gear"": true}, ""raw_score"": 38.08, ""bias"": 56.0, ""sigmoid_slope"": 0.038, ""norm_score"": 33.6}","{""sem_pos_total"": 40.52, ""sem_neg_total"": -46.54, ""highlight_vec_sim"": 0.77, ""highlight_vec_boost"": 0.0, ""amenities_vec_sim"": 0.846, ""amenities_vec_boost"": 10.15, ""keyword_boost_total"": 8.4, ""keyword_hits"": [""superhost:5.6"", ""self check-in:2.8""], ""logic_score"": 6.0, ""density_base_log1p"": 0.0, ""nearby_counts"": {""nightlife"": 0, ""culture"": 0, ""wellness"": 0, ""dining"": 0, ""parks"": 0, ""work_infra"": 0, ""nearby_count"": 0}, ""flags"": {""is_quiet"": true, ""has_wifi"": true, ""has_baby_gear"": true, ""has_romantic_gear"": true}, ""raw_score"": 18.53, ""bias"": 70.0, ""sigmoid_slope"": 0.038, ""norm_score"": 12.4}","{""sem_pos_total"": 39.52, ""sem_neg_total"": -46.4, ""highlight_vec_sim"": 0.619, ""highlight_vec_boost"": 0.0, ""amenities_vec_sim"": 0.661, ""amenities_vec_boost"": 7.93, ""keyword_boost_total"": 21.0, ""keyword_hits"": [""self check-in:15.0"", ""superhost:6.0""], ""logic_score"": 26.0, ""density_base_log1p"": 0.0, ""nearby_counts"": {""nightlife"": 0, ""culture"": 0, ""wellness"": 0, ""dining"": 0, ""parks"": 0, ""work_infra"": 0, ""nearby_count"": 0}, ""flags"": {""is_quiet"": true, ""has_wifi"": true, ""has_baby_gear"": true, ""has_romantic_gear"": true}, ""raw_score"": 48.05, ""bias"": 62.0, ""sigmoid_slope"": 0.038, ""norm_score"": 37.0}",25.82092,-80.33388,0.858794039827322,0.9999303718145105,0.5435872441164183,0.9951260270157359,Couples,15.6,33.6,12.4,37.0,35.3
Paris,"Entire rental unit in Paris, France",https://a0.muscache.com/pictures/hosting/Hosting-15783007/original/1d75dce6-381b-4d36-a104-3d3f7f9797b4.jpeg,https://www.airbnb.ca/rooms/15783007,4,0,4.79,8.6,13.7,19.6,47.9,"24-hour self check-in, Calm and convenient location, Dedicated workspace","{""sem_pos_total"": 40.36, ""sem_neg_total"": -47.77, ""highlight_vec_sim"": 0.85, ""highlight_vec_boost"": 10.21, ""amenities_vec_sim"": 0.64, ""amenities_vec_boost"": 0.0, ""keyword_boost_total"": 7.0, ""keyword_hits"": [""self check-in:7.0""], ""logic_score"": -16.0, ""density_base_log1p"": 0.0, ""nearby_counts"": {""nightlife"": 0, ""culture"": 0,

dominant_persona,count
Singles,4133
Families,4254
Couples,2889
Remote Workers,3087


summary,persona_gap_raw
count,14363
mean,9.487711480888413
stddev,5.0078911485267055
min,0.29999999999999893
25%,6.0
50%,8.5
75%,11.799999999999999
max,44.9
